 Dataset Analysis — Computational Analysis

This notebook contains the analysis for the grapevine dataset:
- label parsing and dataset construction
- image-quality measurements
- duplicate detection
- sharpness analysis
- BBCH/date/stage statistics
- train/validation splits
- DINOv2 embedding extraction
- classification and split comparisons
- early-stage (BBCH 00–19) analysis

The notebook writes intermediate and final results to `DATASET_DIR`, so the explanatory notebook can read them without rerunning the expensive analysis.
The notebook creates the clean dataset as `dataset_clean.csv` (and the duplicate-free dataset as `dataset_clean_no_duplicates.csv`). All other analysis-result CSV files are saved inside `analysis_csv/`.

In [1]:
from pathlib import Path
import re
import hashlib
import json

import pandas as pd
import numpy as np
from PIL import Image
import cv2

DATASET_DIR = Path(
    r"C:/Users/USER/Downloads/clienti-20260916T131257Z-1-001/clienti"
)

# Folder for all analysis CSV outputs
CSV_OUTPUT_DIR = DATASET_DIR / "analysis_csv"
CSV_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

VALID_BBCH = {
    "00", "01", "03", "04", "05", "06",
    "10", "11", "12", "13", "14", "15", "16",
    "51", "53", "55", "56", "57", "59",
    "61", "63", "67", "69",
    "71", "73", "75", "77", "79",
    "81", "83", "85", "87", "89",
    "91", "92", "95"
}

In [2]:
# 1. Build the initial dataset from image filenames and labels

image_files = []
for ext in ["*.jpg", "*.JPG", "*.jpeg", "*.JPEG"]:
    image_files.extend(DATASET_DIR.rglob(ext))

records = []

for image_path in image_files:
    filename = image_path.name
    stem_clean = re.sub(r"\(\d+\)$", "", image_path.stem)
    match = re.search(r"_([^_]*)$", stem_clean)
    raw_label = match.group(1).strip() if match else ""

    if raw_label in VALID_BBCH:
        bbch = int(raw_label)
        label_status = "valid_bbch"
    elif raw_label.upper() == "NC":
        bbch = None
        label_status = "NC"
    else:
        bbch = None
        label_status = "other"

    records.append({
        "filename": filename,
        "path": str(image_path),
        "raw_label": raw_label,
        "bbch": bbch,
        "label_status": label_status
    })

df = pd.DataFrame(records)
df.to_csv(DATASET_DIR / "dataset_clean.csv", index=False)


In [3]:
# 2. Image-quality analysis

CSV_FILE = DATASET_DIR / "dataset_clean.csv"
df = pd.read_csv(CSV_FILE)

def analyze_image(image_path):
    result = {
        "width": None,
        "height": None,
        "aspect_ratio": None,
        "file_size_kb": None,
        "blur_score": None,
        "brightness_mean": None,
        "brightness_std": None,
        "contrast": None,
        "corrupt": False,
        "sha256": None
    }

    try:
        result["file_size_kb"] = image_path.stat().st_size / 1024
        img = Image.open(image_path)
        img.load()

        width, height = img.size
        result["width"] = width
        result["height"] = height
        result["aspect_ratio"] = width / height if height > 0 else None

        gray = np.array(img.convert("L"))
        result["brightness_mean"] = float(gray.mean())
        result["brightness_std"] = float(gray.std())
        result["contrast"] = float(gray.std())
        result["blur_score"] = float(
            cv2.Laplacian(gray, cv2.CV_64F).var()
        )

        sha256 = hashlib.sha256()
        with open(image_path, "rb") as f:
            for chunk in iter(lambda: f.read(8192), b""):
                sha256.update(chunk)
        result["sha256"] = sha256.hexdigest()

    except Exception:
        result["corrupt"] = True

    return result

quality_results = [
    analyze_image(Path(row["path"]))
    for _, row in df.iterrows()
]

quality_df = pd.DataFrame(quality_results)
quality_df = pd.concat(
    [df.reset_index(drop=True), quality_df],
    axis=1
)

def quality_flag(row):
    flags = []

    if row["corrupt"]:
        return "CORRUPT"

    if row["width"] < 500 or row["height"] < 500:
        flags.append("LOW_RESOLUTION")
    if row["aspect_ratio"] < 0.5 or row["aspect_ratio"] > 2.0:
        flags.append("UNUSUAL_ASPECT_RATIO")
    if row["blur_score"] < 50:
        flags.append("VERY_BLURRY")
    elif row["blur_score"] < 150:
        flags.append("BLURRY")

    if row["brightness_mean"] < 40:
        flags.append("VERY_DARK")
    elif row["brightness_mean"] < 60:
        flags.append("DARK")

    if row["brightness_mean"] > 220:
        flags.append("VERY_BRIGHT")
    elif row["brightness_mean"] > 200:
        flags.append("BRIGHT")

    if row["contrast"] < 20:
        flags.append("LOW_CONTRAST")

    return ";".join(flags) if flags else "OK"

quality_df["quality_flag"] = quality_df.apply(quality_flag, axis=1)
quality_df.to_csv(CSV_OUTPUT_DIR / "image_quality_report.csv", index=False)


In [4]:
# 3. Remove duplicate paths and exact duplicate images

df_unique_paths = quality_df.drop_duplicates(subset="path").copy()
df_no_duplicates = df_unique_paths.drop_duplicates(
    subset="sha256", keep="first"
).copy()

# This is the cleaned dataset used by the analysis.
df = df_no_duplicates.copy()
df.to_csv(DATASET_DIR / "dataset_clean_no_duplicates.csv", index=False)

duplicate_summary = pd.DataFrame({
    "quantity": [
        len(quality_df),
        len(df_unique_paths),
        len(df_no_duplicates),
        len(quality_df) - len(df_no_duplicates)
    ]
}, index=[
    "initial_rows",
    "unique_paths",
    "unique_exact_images",
    "exact_duplicates_removed"
])
duplicate_summary.to_csv(DATASET_DIR / "duplicate_summary.csv")


In [5]:
# 4. Label statistics

label_status_counts = (
    df["label_status"]
    .value_counts(dropna=False)
    .rename_axis("label_status")
    .reset_index(name="images")
)
label_status_counts.to_csv(
    CSV_OUTPUT_DIR / "label_status_counts.csv", index=False
)

other_labels = (
    df[df["label_status"] == "other"]["raw_label"]
    .value_counts(dropna=False)
    .rename_axis("raw_label")
    .reset_index(name="images")
)
other_labels.to_csv(
    CSV_OUTPUT_DIR / "other_raw_labels.csv", index=False
)

bbch_distribution = (
    df[df["label_status"] == "valid_bbch"]["bbch"]
    .value_counts()
    .sort_index()
    .rename_axis("bbch")
    .reset_index(name="images")
)
bbch_distribution.to_csv(
    CSV_OUTPUT_DIR / "bbch_distribution.csv", index=False
)


In [6]:
# 5. Sharpness analysis

def calculate_sharpness(path):
    try:
        img = Image.open(path).convert("L")
        gray = np.array(img)

        laplacian = cv2.Laplacian(gray, cv2.CV_64F)
        lap_var = laplacian.var()

        sobel_x = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
        sobel_y = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
        tenengrad = np.mean(sobel_x**2 + sobel_y**2)

        edges = cv2.Canny(gray, 100, 200)
        edge_density = np.mean(edges > 0)

        return lap_var, tenengrad, edge_density
    except Exception:
        return np.nan, np.nan, np.nan

sharpness_results = [
    calculate_sharpness(path)
    for path in df["path"]
]

df["laplacian_var"] = [x[0] for x in sharpness_results]
df["tenengrad"] = [x[1] for x in sharpness_results]
df["edge_density"] = [x[2] for x in sharpness_results]

df["lap_rank"] = df["laplacian_var"].rank(pct=True)
df["tenengrad_rank"] = df["tenengrad"].rank(pct=True)
df["edge_rank"] = df["edge_density"].rank(pct=True)
df["overall_sharpness_rank"] = (
    df["lap_rank"] + df["tenengrad_rank"] + df["edge_rank"]
) / 3

df.to_csv(CSV_OUTPUT_DIR / "dataset_analysis_with_sharpness.csv", index=False)

sharpness_statistics = df[
    ["blur_score", "laplacian_var", "tenengrad", "edge_density"]
].describe()
sharpness_statistics.to_csv(
    CSV_OUTPUT_DIR / "sharpness_statistics.csv"
)

difficult = df.sort_values("overall_sharpness_rank").head(30)
difficult.to_csv(
    CSV_OUTPUT_DIR / "lowest_sharpness_images.csv", index=False
)


In [7]:
# 6. BBCH-specific analysis

bbch_df = df[df["label_status"] == "valid_bbch"].copy()

blur_by_stage = (
    bbch_df
    .groupby("bbch")["blur_score"]
    .agg(["count", "mean", "median", "min", "max"])
    .sort_index()
)
blur_by_stage.to_csv(CSV_OUTPUT_DIR / "blur_by_bbch_stage.csv")

class_counts = (
    bbch_df["bbch"]
    .value_counts()
    .sort_index()
    .rename_axis("bbch")
    .reset_index(name="images")
)
class_counts["percentage"] = (
    class_counts["images"] / class_counts["images"].sum() * 100
).round(2)
class_counts["imbalance_ratio_vs_largest"] = (
    class_counts["images"] / class_counts["images"].max()
).round(3)
class_counts.to_csv(
    CSV_OUTPUT_DIR / "bbch_class_balance.csv", index=False
)


In [8]:
# 7. Date analysis

def extract_filename_date(path):
    filename = Path(path).name
    match = re.search(r"(20\d{6})", filename)
    return match.group(1) if match else None

bbch_df["date"] = bbch_df["path"].apply(extract_filename_date)
bbch_df["date_dt"] = pd.to_datetime(
    bbch_df["date"], format="%Y%m%d", errors="coerce"
)
bbch_df["year"] = bbch_df["date_dt"].dt.year

date_stage_counts = pd.crosstab(
    bbch_df["date_dt"], bbch_df["bbch"]
).sort_index()
date_stage_counts.to_csv(
    CSV_OUTPUT_DIR / "bbch_stage_counts_by_date.csv"
)

date_summary = (
    bbch_df.groupby(["date_dt", "bbch"])
    .size()
    .reset_index(name="images")
    .sort_values(["date_dt", "bbch"])
)
date_summary.to_csv(
    CSV_OUTPUT_DIR / "bbch_date_summary_long.csv", index=False
)

year_stage_summary = (
    bbch_df.groupby(["year", "bbch"])
    .agg(
        images=("filename", "size"),
        first_date=("date_dt", "min"),
        last_date=("date_dt", "max")
    )
    .reset_index()
    .sort_values(["year", "bbch"])
)
year_stage_summary.to_csv(
    CSV_OUTPUT_DIR / "bbch_stage_summary_by_year.csv", index=False
)

date_missing = bbch_df[bbch_df["date_dt"].isna()][
    ["filename", "path", "bbch"
]
]
date_missing.to_csv(
    CSV_OUTPUT_DIR / "bbch_missing_dates.csv", index=False
)


In [9]:
# 8. Map BBCH values to broader growth-stage groups

def assign_growth_stage(bbch):
    if 0 <= bbch <= 19:
        return "early_development"
    if 51 <= bbch <= 69:
        return "flowering"
    if 71 <= bbch <= 79:
        return "fruit_development"
    if 81 <= bbch <= 95:
        return "ripening"
    return "unassigned"

model_df = bbch_df.copy()
model_df["growth_stage"] = model_df["bbch"].apply(assign_growth_stage)

stage_counts = (
    model_df["growth_stage"]
    .value_counts()
    .rename_axis("growth_stage")
    .reset_index(name="images")
)
stage_counts["percentage"] = (
    stage_counts["images"] / stage_counts["images"].sum() * 100
).round(2)

model_df.to_csv(
    CSV_OUTPUT_DIR / "dataset_growth_stage_labels.csv", index=False
)
stage_counts.to_csv(
    CSV_OUTPUT_DIR / "growth_stage_balance.csv", index=False
)


In [10]:
# 9. Random train/validation split

from sklearn.model_selection import train_test_split

train_df, validation_df = train_test_split(
    model_df,
    test_size=0.20,
    random_state=42,
    stratify=model_df["growth_stage"]
)

train_df = train_df.sort_index().reset_index(drop=True)
validation_df = validation_df.sort_index().reset_index(drop=True)

train_df.to_csv(
    CSV_OUTPUT_DIR / "growth_stage_train.csv", index=False
)
validation_df.to_csv(
    CSV_OUTPUT_DIR / "growth_stage_validation.csv", index=False
)


In [11]:
# 10. DINOv2 embeddings + baseline classifier

import torch
from torchvision import transforms
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix
)

DINO_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

DINO_MODEL = torch.hub.load(
    "facebookresearch/dinov2",
    "dinov2_vits14"
).to(DINO_DEVICE)
DINO_MODEL.eval()

DINO_TRANSFORM = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    )
])

def extract_dinov2_embeddings(dataframe):
    embeddings = []
    labels = []

    with torch.inference_mode():
        for _, row in dataframe.iterrows():
            image = Image.open(row["path"]).convert("RGB")
            image_tensor = DINO_TRANSFORM(image).unsqueeze(0).to(DINO_DEVICE)
            embedding = DINO_MODEL(image_tensor)
            embeddings.append(embedding.squeeze(0).cpu().numpy())
            labels.append(row["growth_stage"])

    return np.asarray(embeddings), np.asarray(labels)

X_train, y_train = extract_dinov2_embeddings(train_df)
X_validation, y_validation = extract_dinov2_embeddings(validation_df)

classifier = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    random_state=42
)
classifier.fit(X_train, y_train)
y_validation_pred = classifier.predict(X_validation)

baseline_metrics = {
    "accuracy": accuracy_score(y_validation, y_validation_pred),
    "balanced_accuracy": balanced_accuracy_score(
        y_validation, y_validation_pred
    )
}

with open(DATASET_DIR / "dinov2_baseline_metrics.json", "w") as f:
    json.dump(baseline_metrics, f, indent=2)

label_order = sorted(model_df["growth_stage"].unique())
confusion = pd.DataFrame(
    confusion_matrix(
        y_validation,
        y_validation_pred,
        labels=label_order
    ),
    index=label_order,
    columns=label_order
)
confusion.to_csv(
    CSV_OUTPUT_DIR / "dinov2_growth_stage_confusion_matrix.csv"
)

np.savez_compressed(
    DATASET_DIR / "dinov2_embeddings.npz",
    X_train=X_train,
    y_train=y_train,
    X_validation=X_validation,
    y_validation=y_validation
)


Using cache found in C:\Users\USER/.cache\torch\hub\facebookresearch_dinov2_main
C:\Users\USER/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
C:\Users\USER/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
C:\Users\USER/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


In [12]:
# 11. Repeated stratified splits

from sklearn.model_selection import StratifiedShuffleSplit

X_all, y_all = extract_dinov2_embeddings(model_df)

splitter = StratifiedShuffleSplit(
    n_splits=5,
    test_size=0.20,
    random_state=42
)

split_results = []

for split_number, (train_indices, validation_indices) in enumerate(
    splitter.split(X_all, y_all),
    start=1
):
    split_classifier = LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    )
    split_classifier.fit(X_all[train_indices], y_all[train_indices])
    split_predictions = split_classifier.predict(
        X_all[validation_indices]
    )

    split_results.append({
        "split": split_number,
        "accuracy": accuracy_score(
            y_all[validation_indices], split_predictions
        ),
        "balanced_accuracy": balanced_accuracy_score(
            y_all[validation_indices], split_predictions
        )
    })

split_results_df = pd.DataFrame(split_results)
split_results_df.to_csv(
    CSV_OUTPUT_DIR / "dinov2_repeated_split_results.csv", index=False
)


In [13]:
# 12. Date-held-out split

chronological_df = model_df.copy()
chronological_df["date_dt"] = pd.to_datetime(
    chronological_df["date"],
    format="%Y%m%d",
    errors="coerce"
)
chronological_df = chronological_df.dropna(subset=["date_dt"]).copy()

unique_dates = sorted(chronological_df["date_dt"].unique())
validation_date_count = max(
    1, int(round(len(unique_dates) * 0.20))
)
validation_dates = set(unique_dates[-validation_date_count:])

chronological_train_df = chronological_df[
    ~chronological_df["date_dt"].isin(validation_dates)
].copy()
chronological_validation_df = chronological_df[
    chronological_df["date_dt"].isin(validation_dates)
].copy()

index_to_embedding = pd.Series(
    np.arange(len(model_df)), index=model_df.index
)

train_indices = index_to_embedding.loc[
    chronological_train_df.index
].to_numpy()
validation_indices = index_to_embedding.loc[
    chronological_validation_df.index
].to_numpy()

chronological_classifier = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    random_state=42
)
chronological_classifier.fit(
    X_all[train_indices], y_all[train_indices]
)
chronological_predictions = chronological_classifier.predict(
    X_all[validation_indices]
)
chronological_targets = y_all[validation_indices]

chronological_metrics = {
    "accuracy": accuracy_score(
        chronological_targets, chronological_predictions
    ),
    "balanced_accuracy": balanced_accuracy_score(
        chronological_targets, chronological_predictions
    ),
    "training_images": len(chronological_train_df),
    "validation_images": len(chronological_validation_df)
}

with open(DATASET_DIR / "dinov2_date_heldout_metrics.json", "w") as f:
    json.dump(chronological_metrics, f, indent=2)

pd.DataFrame(
    confusion_matrix(
        chronological_targets,
        chronological_predictions,
        labels=label_order
    ),
    index=label_order,
    columns=label_order
).to_csv(
    CSV_OUTPUT_DIR / "dinov2_date_heldout_confusion_matrix.csv"
)


In [14]:
# 13. One latest date held out from each growth stage

balanced_date_df = model_df.copy()
balanced_date_df["date_dt"] = pd.to_datetime(
    balanced_date_df["date"],
    format="%Y%m%d",
    errors="coerce"
)

latest_date_by_stage = (
    balanced_date_df
    .dropna(subset=["date_dt"])
    .groupby("growth_stage")["date_dt"]
    .max()
)

balanced_validation_dates = set(latest_date_by_stage.tolist())
balanced_validation_mask = balanced_date_df["date_dt"].isin(
    balanced_validation_dates
)

balanced_train_df = balanced_date_df[~balanced_validation_mask].copy()
balanced_validation_df = balanced_date_df[balanced_validation_mask].copy()

balanced_train_indices = index_to_embedding.loc[
    balanced_train_df.index
].to_numpy()
balanced_validation_indices = index_to_embedding.loc[
    balanced_validation_df.index
].to_numpy()

balanced_classifier = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    random_state=42
)
balanced_classifier.fit(
    X_all[balanced_train_indices],
    y_all[balanced_train_indices]
)

balanced_predictions = balanced_classifier.predict(
    X_all[balanced_validation_indices]
)
balanced_targets = y_all[balanced_validation_indices]

balanced_metrics = {
    "accuracy": accuracy_score(
        balanced_targets, balanced_predictions
    ),
    "balanced_accuracy": balanced_accuracy_score(
        balanced_targets, balanced_predictions
    ),
    "training_images": len(balanced_train_df),
    "validation_images": len(balanced_validation_df)
}

with open(
    DATASET_DIR / "dinov2_balanced_date_heldout_metrics.json", "w"
) as f:
    json.dump(balanced_metrics, f, indent=2)

pd.DataFrame(
    confusion_matrix(
        balanced_targets,
        balanced_predictions,
        labels=label_order
    ),
    index=label_order,
    columns=label_order
).to_csv(
    CSV_OUTPUT_DIR / "dinov2_balanced_date_confusion_matrix.csv"
)

balanced_train_df.to_csv(
    CSV_OUTPUT_DIR / "growth_stage_balanced_date_train.csv", index=False
)
balanced_validation_df.to_csv(
    CSV_OUTPUT_DIR / "growth_stage_balanced_date_validation.csv", index=False
)


In [15]:
# 14. Image-only / date-only / image+date comparison and late fusion

from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

aligned_model_df = model_df.reset_index(drop=True).copy()
aligned_model_df["date_dt"] = pd.to_datetime(
    aligned_model_df["date"],
    format="%Y%m%d",
    errors="coerce"
)

date_missing = aligned_model_df["date_dt"].isna().astype(float).to_numpy()
day_of_year = aligned_model_df["date_dt"].dt.dayofyear.fillna(0).to_numpy()

date_features = np.column_stack([
    np.where(
        date_missing == 1, 0,
        np.sin(2 * np.pi * day_of_year / 365.25)
    ),
    np.where(
        date_missing == 1, 0,
        np.cos(2 * np.pi * day_of_year / 365.25)
    ),
    date_missing
])

valid_embeddings = X_all
valid_labels = y_all

random_train_indices, random_validation_indices = train_test_split(
    np.arange(len(aligned_model_df)),
    test_size=0.20,
    random_state=42,
    stratify=valid_labels
)

latest_date_by_stage = (
    aligned_model_df
    .dropna(subset=["date_dt"])
    .groupby("growth_stage")["date_dt"]
    .max()
)
balanced_dates = set(latest_date_by_stage.tolist())
balanced_validation_mask = aligned_model_df["date_dt"].isin(
    balanced_dates
).to_numpy()
balanced_train_indices = np.where(~balanced_validation_mask)[0]
balanced_validation_indices = np.where(balanced_validation_mask)[0]

def evaluate_feature_sets(train_indices, validation_indices, split_name):
    targets = valid_labels[validation_indices]
    results = []

    feature_sets = {
        "image_only": (
            valid_embeddings[train_indices],
            valid_embeddings[validation_indices]
        ),
        "date_only": (
            date_features[train_indices],
            date_features[validation_indices]
        ),
        "image_plus_date": (
            np.column_stack([
                valid_embeddings[train_indices],
                date_features[train_indices]
            ]),
            np.column_stack([
                valid_embeddings[validation_indices],
                date_features[validation_indices]
            ])
        )
    }

    for model_name, (train_features, validation_features) in feature_sets.items():
        model = make_pipeline(
            StandardScaler(),
            LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=42
            )
        )
        model.fit(train_features, valid_labels[train_indices])
        predictions = model.predict(validation_features)

        results.append({
            "split": split_name,
            "model": model_name,
            "accuracy": accuracy_score(targets, predictions),
            "balanced_accuracy": balanced_accuracy_score(
                targets, predictions
            )
        })

    return results

comparison_results = evaluate_feature_sets(
    random_train_indices,
    random_validation_indices,
    "random"
)
comparison_results.extend(
    evaluate_feature_sets(
        balanced_train_indices,
        balanced_validation_indices,
        "phase_balanced_date_heldout"
    )
)

comparison_df = pd.DataFrame(comparison_results)
comparison_df.to_csv(
    CSV_OUTPUT_DIR / "dinov2_image_date_comparison.csv", index=False
)

def evaluate_late_fusion(train_indices, validation_indices, split_name):
    image_model = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=42
        )
    )
    date_model = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=42
        )
    )

    image_model.fit(
        valid_embeddings[train_indices],
        valid_labels[train_indices]
    )
    date_model.fit(
        date_features[train_indices],
        valid_labels[train_indices]
    )

    image_probabilities = image_model.predict_proba(
        valid_embeddings[validation_indices]
    )
    date_probabilities = date_model.predict_proba(
        date_features[validation_indices]
    )

    image_probability_df = pd.DataFrame(
        image_probabilities,
        columns=image_model.classes_
    ).reindex(columns=label_order)
    date_probability_df = pd.DataFrame(
        date_probabilities,
        columns=date_model.classes_
    ).reindex(columns=label_order)

    image_log_scores = np.log(
        np.clip(image_probability_df.to_numpy(), 1e-8, 1.0)
    )
    date_log_scores = np.log(
        np.clip(date_probability_df.to_numpy(), 1e-8, 1.0)
    )

    results = []
    targets = valid_labels[validation_indices]

    for image_weight in [0.9, 0.8, 0.7]:
        combined_scores = (
            image_weight * image_log_scores
            + (1 - image_weight) * date_log_scores
        )
        predictions = np.asarray(label_order)[
            np.argmax(combined_scores, axis=1)
        ]

        results.append({
            "split": split_name,
            "image_weight": image_weight,
            "date_weight": round(1 - image_weight, 2),
            "accuracy": accuracy_score(targets, predictions),
            "balanced_accuracy": balanced_accuracy_score(
                targets, predictions
            )
        })

    return results

late_fusion_results = evaluate_late_fusion(
    random_train_indices,
    random_validation_indices,
    "random"
)
late_fusion_results.extend(
    evaluate_late_fusion(
        balanced_train_indices,
        balanced_validation_indices,
        "phase_balanced_date_heldout"
    )
)

late_fusion_df = pd.DataFrame(late_fusion_results)
late_fusion_df.to_csv(
    CSV_OUTPUT_DIR / "dinov2_image_dominant_late_fusion.csv",
    index=False
)


In [16]:
# 15. Early phenological stages: BBCH 00–19

early_df = bbch_df[bbch_df["bbch"].between(0, 19)].copy()

early_resolution = early_df[["width", "height"]].describe()
early_resolution.to_csv(
    CSV_OUTPUT_DIR / "early_bbch_00_19_resolution_statistics.csv"
)

early_unique_resolutions = (
    early_df.groupby(["width", "height"])
    .size()
    .sort_values(ascending=False)
    .head(30)
    .rename("images")
    .reset_index()
)
early_unique_resolutions.to_csv(
    CSV_OUTPUT_DIR / "early_bbch_00_19_unique_resolutions.csv",
    index=False
)

early_blur_statistics = early_df["blur_score"].describe()
early_blur_statistics.to_csv(
    CSV_OUTPUT_DIR / "early_bbch_00_19_blur_statistics.csv"
)

early_by_stage = (
    early_df.groupby("bbch")
    .agg(
        images=("filename", "count"),
        min_width=("width", "min"),
        median_width=("width", "median"),
        min_height=("height", "min"),
        median_height=("height", "median"),
        median_blur=("blur_score", "median"),
        min_blur=("blur_score", "min")
    )
    .sort_index()
)
early_by_stage.to_csv(
    CSV_OUTPUT_DIR / "early_bbch_00_19_by_stage.csv"
)

early_image_details = early_df[
    [
        "filename", "bbch", "width", "height",
        "blur_score", "quality_flag"
    ]
].sort_values(["bbch", "width"])
early_image_details.to_csv(
    CSV_OUTPUT_DIR / "early_bbch_00_19_image_details.csv",
    index=False
)
